# CC1A screening recall

In [1]:
import ast
from pathlib import Path

import pandas as pd

In [2]:
annotated_path = Path("../data/consistency-check-datasets/screening/annotated_screening_dataset.xlsx")
reference_path = Path("../data/labels/l0/L0) Screening.csv")

human = pd.read_excel(annotated_path, sheet_name="l0-sample")
expert = pd.read_csv(reference_path)

In [3]:
human["custom_id"] = human["custom_id"].astype(str).str.strip()
expert_labels = expert[["UT (Unique WOS ID)", "eligbility"]].copy()
expert_labels["UT (Unique WOS ID)"] = expert_labels["UT (Unique WOS ID)"].astype(str).str.strip()
expert_labels = expert_labels.drop_duplicates(subset=["UT (Unique WOS ID)", "eligbility"])
expert_labels = expert_labels.drop_duplicates(subset=["UT (Unique WOS ID)"], keep="first")

df = human.merge(
    expert_labels,
    left_on="custom_id",
    right_on="UT (Unique WOS ID)",
    how="left",
    validate="one_to_one",
)

len(df), df["eligbility"].isna().sum()

(100, np.int64(0))

In [4]:
def normalize_label(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list) and parsed:
            text = str(parsed[0])
    except Exception:
        pass
    return text.strip().strip('"').upper()

df["human_label"] = df["eligibility"].map(normalize_label)
df["expert_label"] = df["eligbility"].map(normalize_label)
df[["human_label", "expert_label"]].value_counts().reset_index(name="count")

,human_label,expert_label,count
0,ELIGIBLE,ELIGIBLE,83
1,NOT_ELIGIBLE,ELIGIBLE,17


In [5]:
scored = df[df["expert_label"] != ""].copy()
tp = int(((scored["human_label"] == "ELIGIBLE") & (scored["expert_label"] == "ELIGIBLE")).sum())
fn = int(((scored["human_label"] != "ELIGIBLE") & (scored["expert_label"] == "ELIGIBLE")).sum())
fp = int(((scored["human_label"] == "ELIGIBLE") & (scored["expert_label"] != "ELIGIBLE")).sum())
tn = int(((scored["human_label"] != "ELIGIBLE") & (scored["expert_label"] != "ELIGIBLE")).sum())
recall = tp / (tp + fn) if (tp + fn) else 0.0

pd.DataFrame([
    {
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
        "positive_support": tp + fn,
        "scored_rows": len(scored),
        "eligible_recall": recall,
    }
])

,tp,fn,fp,tn,positive_support,scored_rows,eligible_recall
0,83,17,0,0,100,100,0.83


In [6]:
df.loc[df["human_label"] != df["expert_label"], ["Article Title", "human_label", "expert_label", "inclusion_reason", "exclusion_reason"]]

,Article Title,human_label,expert_label,inclusion_reason,exclusion_reason
83,Scientific Foundations for an IUCN Red List of...,NOT_ELIGIBLE,ELIGIBLE,NaN,not related to any specific biodiversity chang...
84,A review of spatial patterns across species ra...,NOT_ELIGIBLE,ELIGIBLE,NaN,Text is a review of spatial patterns for conse...
85,Using restriction fragment length polymorphism...,NOT_ELIGIBLE,ELIGIBLE,NaN,unclear what is biodiversity change here
86,Metapopulation structure of the specialized he...,NOT_ELIGIBLE,ELIGIBLE,NaN,can't infer what is anthropogenic driver here
87,Population genetic structure of the Japanese e...,NOT_ELIGIBLE,ELIGIBLE,NaN,can't infer what is anthropogenic driver here
88,Population genetic structure of dolphinfish (C...,NOT_ELIGIBLE,ELIGIBLE,NaN,can't infer what is anthropogenic driver here
89,Inferring the ancestry of African wild dogs th...,NOT_ELIGIBLE,ELIGIBLE,NaN,unclear about the anthropogenic driver here. H...
90,Attributing carbon changes in conterminous U.S...,NOT_ELIGIBLE,ELIGIBLE,NaN,Text is not about biodiversity change but carb...
91,Landscape genetics of an early successional sp...,NOT_ELIGIBLE,ELIGIBLE,NaN,unclear what is biodiversity change here
92,Genetic Network and Breeding Patterns of a Sic...,NOT_ELIGIBLE,ELIGIBLE,NaN,text is describing population with limited gen...
